# NB3 — FastAPI `/search` Endpoint + Latency Benchmark

**Stack:** FastAPI + uvicorn + httpx (client). Searcher từ `app/search.py`.
Maps to slide §7 (Production Patterns) + deliverable bullets 1, 4.

> Mục tiêu: bọc `Searcher` thành REST API, đo P50/P95/P99 latency, đảm bảo
> P99 < 50 ms cho hybrid mode (rubric threshold).

In [1]:
import _setup  # noqa: F401
import json
import statistics
import subprocess
import sys
import time
from pathlib import Path

import httpx

## 1. Khởi động API server (background)

Trong production thực tế, bạn sẽ chạy `make api` ở terminal riêng. Notebook
này khởi động uvicorn ở background subprocess và đợi `/healthz` trả ready.

In [2]:
ROOT = Path(_setup.__file__).resolve().parent.parent
URL = "http://localhost:8000"
http = httpx.Client(base_url=URL, timeout=10.0)
proc = None
try:
    r = http.get("/healthz")
    already_ready = r.status_code == 200 and r.json().get("ready")
except httpx.HTTPError:
    already_ready = False
if not already_ready:
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app.main:app", "--port", "8000", "--log-level", "warning"],
        cwd=str(ROOT),
    )

# Đợi server up + warm (Searcher.from_corpus loads embeddings + indexes 1000 docs)
for _ in range(180):
    try:
        r = http.get("/healthz", timeout=2.0)
        if r.status_code == 200 and r.json().get("ready"):
            break
    except httpx.HTTPError:
        pass
    time.sleep(1)
else:
    raise RuntimeError("API didn't become ready within 180s")

print(http.get("/healthz").json())

{'ready': True, 'n_docs': 1000}


## 2. Single query — kiểm tra response shape

In [3]:
r = http.get("/search", params={"q": "cloud computing tự động mở rộng", "mode": "hybrid"})
r.raise_for_status()
body = r.json()
assert {"query", "mode", "top_k", "latency_ms", "hits"} <= body.keys()
print(json.dumps({**body, "hits": body["hits"][:3]}, ensure_ascii=False, indent=2))
print(f"latency_ms: {body['latency_ms']:.1f}")
print(f"top-3 hits:")
for h in body["hits"][:3]:
    print(f"  {h['doc_id']:>14}  score={h['score']:.4f}  {h['title']}")

{
  "query": "cloud computing tự động mở rộng",
  "mode": "hybrid",
  "top_k": 10,
  "latency_ms": 8.669200000440469,
  "hits": [
    {
      "doc_id": "cloud_016",
      "title": "Điện toán đám mây: tự động mở rộng theo lưu lượng",
      "text": "Theo khuyến nghị 2026, điện toán đám mây đòi hỏi phải tự động mở rộng theo lưu lượng. Các công cụ phổ biến gồm cloud, serverless, AWS. Khái niệm hạ tầng và không máy chủ là nền tảng quan trọng. Bài viết này giải thích cách tự động mở rộng theo lưu lượng một cách thực tế, kèm ví dụ minh hoạ với cloud và AWS.",
      "score": 0.03252247488101534
    },
    {
      "doc_id": "cloud_072",
      "title": "Điện toán đám mây: tự động mở rộng theo lưu lượng",
      "text": "Theo khuyến nghị 2026, điện toán đám mây đòi hỏi phải tự động mở rộng theo lưu lượng. Các công cụ phổ biến gồm cloud, Azure, Kubernetes. Khái niệm triển khai và đám mây là nền tảng quan trọng. Bài viết này giải thích cách tự động mở rộng theo lưu lượng một cách thực tế, kèm ví dụ 

## 3. Latency benchmark (100 queries × 3 modes)

Dùng 50 golden queries × 2 reps = 100 calls/mode. Ghi nhận latency từ
`body["latency_ms"]` (server-side, đã trừ network) HOẶC từ wall-clock httpx
(bao gồm network) — note: rubric assert P99 < 50ms áp dụng cho server-side.

Output: bảng P50/P95/P99 cho 3 mode.

In [4]:
DATA = ROOT / "data"
golden = [json.loads(l) for l in (DATA / "golden_set.jsonl").open(encoding="utf-8")]

# Warm up all three retrieval modes before measuring server-side latency.
for mode in ("keyword", "semantic", "hybrid"):
    for q in golden[:10]:
        r = http.get("/search", params={"q": q["query"], "mode": mode})
        r.raise_for_status()


def percentile(values: list[float], p: float) -> float:
    n = len(values)
    if n == 0:
        return 0.0
    return sorted(values)[min(int(n * p), n - 1)]


def benchmark_mode(mode: str, reps: int = 2) -> dict[str, float]:
    server_latencies: list[float] = []
    wall_latencies: list[float] = []
    for _ in range(reps):
        for q in golden:
            t0 = time.perf_counter()
            r = http.get("/search", params={"q": q["query"], "mode": mode})
            r.raise_for_status()
            wall_latencies.append((time.perf_counter() - t0) * 1000)
            server_latencies.append(r.json()["latency_ms"])
    return {
        "p50_server": percentile(server_latencies, 0.50),
        "p95_server": percentile(server_latencies, 0.95),
        "p99_server": percentile(server_latencies, 0.99),
        "p99_wall":   percentile(wall_latencies, 0.99),
    }


print(f"  {'mode':10}  {'P50':>7}  {'P95':>7}  {'P99':>7}  {'P99(wall)':>9}")
results = {}
for mode in ("keyword", "semantic", "hybrid"):
    res = benchmark_mode(mode)
    results[mode] = res
    print(f"  {mode:10}  {res['p50_server']:>5.1f}ms  {res['p95_server']:>5.1f}ms  "
          f"{res['p99_server']:>5.1f}ms  {res['p99_wall']:>7.1f}ms")

  mode            P50      P95      P99  P99(wall)


  keyword       1.3ms    2.5ms    3.3ms      5.0ms


  semantic      8.5ms   10.7ms   15.2ms     17.8ms


  hybrid       10.5ms   14.2ms   16.0ms     18.2ms


## 4. Rubric assertion — hybrid P99 server-side < 50ms

In [5]:
hybrid_p99 = results["hybrid"]["p99_server"]
print(f"Hybrid P99 server-side: {hybrid_p99:.1f}ms")
if hybrid_p99 < 50:
    print(f"PASS — hybrid P99 < 50ms ({hybrid_p99:.1f}ms)")
else:
    print(f"WARN — hybrid P99 >= 50ms ({hybrid_p99:.1f}ms)")
    print("  Possible causes: cold cache, fastembed model not warm yet, or RRF depth=50 is too aggressive")
    print("  Check: re-run benchmark after 10 warm-up queries; or reduce RRF depth")

Hybrid P99 server-side: 16.0ms
PASS — hybrid P99 < 50ms (16.0ms)


## 5. Cleanup — stop the API server

In [6]:
http.close()
if proc is not None:
    proc.terminate()
    proc.wait(timeout=5)
    print("API server stopped")
else:
    print("Notebook client closed; existing API server is still running")

API server stopped


## Deliverable evidence

1. Output cell 2: 1 single hybrid query response with `top-3 hits`.
2. Output cell 3: latency table P50/P95/P99 for keyword/semantic/hybrid.
3. Output cell 4: hybrid P99 < 50ms PASS.

---

## Vibe-coding callout

**Delegate freely:** the FastAPI scaffolding (route definition, Pydantic
response model, lifespan handler). AI generates this perfectly given the
spec "GET /search?q=str&mode=Literal[...] returning SearchResponse with
latency_ms field". `app/main.py` is exactly that pattern — review the diff,
don't write it from scratch.

**Think hard yourself:** *what to measure*. Server-side latency vs wall-clock
vs client-side. P50 vs P95 vs P99. Cold vs warm. Single user vs concurrent.
These are *judgement* decisions: nếu rubric chỉ check P99, optimization sẽ
hướng vào tail latency, không phải mean. Đừng nhờ AI quyết định metric —
chỉ nhờ implement metric đã chọn.